# Smart Campus Mess Food Waste Predictor
## Notebook 02: Model Training, Benchmarking & Recipe Optimization
**Author:** Khushi Yadav (B.Tech Computer Science)  
**Objective:** Build and benchmark regression models (Ridge, Random Forest, XGBoost) against naive campus baselines, scale recipes with safety buffers, and evaluate total food waste reduction.

In [1]:
import sys
sys.path.append('..')
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import train_and_benchmark
from src.optimizer import compute_waste_reduction_simulation, scale_recipe_quantities

### 1. Model Training & Out-of-Sample Chronological Benchmarking
We train on the initial 80% time period and evaluate on the forward 20% test period.

In [2]:
winning_model, preprocessor, summary = train_and_benchmark()
benchmarks_df = pd.DataFrame(summary['benchmark_metrics']).T
benchmarks_df

### 2. Feature Importance Breakdown (XGBoost)
Analyzing what factors most strongly dictate hostel mess attendance.

In [3]:
feat_series = pd.Series(summary['top_feature_importance']).sort_values()
plt.figure(figsize=(10, 6))
feat_series.plot(kind='barh', color='teal')
plt.title('Top Predictive Features for Mess Attendance (XGBoost)')
plt.xlabel('Relative Feature Importance')
plt.show()

### 3. Food Waste Reduction & Sustainability Quantification
Comparing food wastage under traditional 85% fixed cooking vs ML-driven predictive cooking with safety buffer.

In [4]:
df_raw = pd.read_csv('../data/mess_attendance_raw.csv')
X_test = preprocessor.transform(df_raw).iloc[-summary['test_samples']:]
y_pred = winning_model.predict(X_test)

sim_results = compute_waste_reduction_simulation(df_raw, y_pred, safety_buffer_pct=0.05)
for k, v in sim_results.items():
    print(f'{k}: {v}')

### 4. Dynamic Recipe Scaling Demonstration
Scaling ingredient batch sizes for a predicted lunch of 620 students.

In [5]:
recipe = scale_recipe_quantities('Lunch', predicted_attendance=620, safety_buffer_pct=0.05)
print(f'Target Headcount: {recipe["buffered_target_headcount"]} portions')
pd.DataFrame(list(recipe['ingredients'].items()), columns=['Ingredient', 'Scaled Batch Quantity'])